# DeePoo YOLOv4-Tiny 416x416 — Training + TFLite INT8 Conversion

End-to-end notebook: train YOLOv4-Tiny with darknet, convert to TFLite INT8 for Android.

**Dataset**: `boxed_640x640.tgz` (YOLO format, 1 class: `poo`, 640x640 images resized to 416x416)  
**Input**: 416x416  
**Output**: TFLite INT8 quantized model  
**Environment**: Google Colab with GPU

## Pipeline

1. Mount Google Drive & extract dataset
2. Resize images 640x640 → 416x416 (labels are normalized, no change needed)
3. Build darknet (AlexeyAB fork)
4. Generate darknet config files (`.cfg`, `.data`, `.names`)
5. Train YOLOv4-Tiny
6. Test darknet model (mAP + visual)
7. Convert darknet weights → TensorFlow → TFLite INT8
8. Validate TFLite model
9. Copy final model to Google Drive

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

import os, sys
print(f"Python: {sys.version}")
print(f"GPU available: {'NVIDIA' in os.popen('nvidia-smi').read()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path
import shutil
import tarfile
import random
import json
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# Project directories
PROJECT_DIR = Path('/content/deepoo-yolov4')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset (640x640 YOLO format, will be resized to 416x416)
DATASET_TGZ = Path('/content/drive/MyDrive/Datasets/DeePoo/boxed_640x640.tgz')
DATASET_DIR_ORIG = Path('/content/boxed_640x640')       # original 640x640
DATASET_DIR = Path('/content/boxed_416x416')             # resized for training

# Output directory on Drive
DRIVE_OUTPUT = Path('/content/drive/MyDrive/Models/yolov4_tiny_deepoo')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

# Model config
CLASS_NAMES = ['poo']
NUM_CLASSES = len(CLASS_NAMES)
INPUT_SIZE = 416
ORIG_SIZE = 640

print(f"Project dir: {PROJECT_DIR}")
print(f"Dataset tgz: {DATASET_TGZ}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Original size: {ORIG_SIZE}x{ORIG_SIZE}")
print(f"Training size: {INPUT_SIZE}x{INPUT_SIZE}")

## 2. Extract Dataset

In [ ]:
# Extract dataset (tgz archive)
if not DATASET_DIR_ORIG.exists():
    if not DATASET_TGZ.exists():
        raise FileNotFoundError(f"Dataset not found: {DATASET_TGZ}")
    print("Extracting dataset...")
    with tarfile.open(DATASET_TGZ, 'r:gz') as tf:
        tf.extractall(DATASET_DIR_ORIG.parent)
    print(f"Extracted to: {DATASET_DIR_ORIG}")
else:
    print(f"Dataset already at: {DATASET_DIR_ORIG}")

# Verify original dataset structure (YOLO format)
orig_train_img = DATASET_DIR_ORIG / 'images' / 'train'
orig_val_img   = DATASET_DIR_ORIG / 'images' / 'val'
orig_train_lbl = DATASET_DIR_ORIG / 'labels' / 'train'
orig_val_lbl   = DATASET_DIR_ORIG / 'labels' / 'val'

print(f"\nOriginal dataset ({ORIG_SIZE}x{ORIG_SIZE}):")
print(f"  Train images: {len(list(orig_train_img.glob('*.jpg')))}")
print(f"  Val images:   {len(list(orig_val_img.glob('*.jpg')))}")

## 3. Resize Images to 416x416

The dataset has 640x640 images. Darknet trains at 416x416 (set in the `.cfg`).  
We resize images now so darknet doesn't have to do it on-the-fly, and so our  
calibration data for INT8 quantization matches the training resolution.

YOLO labels are normalized (0–1), so they remain valid after resizing.

In [ ]:
# Resize images from 640x640 to 416x416 and copy labels
# YOLO labels are normalized so they don't change with resize

def resize_dataset(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir, target_size):
    """Resize images and copy labels to new directory."""
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    img_files = sorted(src_img_dir.glob('*.jpg'))
    for img_path in tqdm(img_files, desc=f"Resizing {src_img_dir.parent.name}/{src_img_dir.name}"):
        # Resize image
        img = Image.open(img_path).convert('RGB')
        img_resized = img.resize((target_size, target_size), Image.LANCZOS)
        img_resized.save(dst_img_dir / img_path.name, quality=95)

        # Copy label (normalized coords don't change)
        lbl_src = src_lbl_dir / img_path.with_suffix('.txt').name
        lbl_dst = dst_lbl_dir / lbl_src.name
        if lbl_src.exists():
            shutil.copy2(lbl_src, lbl_dst)

    return len(img_files)

if not DATASET_DIR.exists():
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    for split in ['train', 'val', 'test']:
        src_img = DATASET_DIR_ORIG / 'images' / split
        src_lbl = DATASET_DIR_ORIG / 'labels' / split
        dst_img = DATASET_DIR / 'images' / split
        dst_lbl = DATASET_DIR / 'labels' / split
        if src_img.exists():
            n = resize_dataset(src_img, src_lbl, dst_img, dst_lbl, INPUT_SIZE)
            print(f"  [{split}] Resized {n} images to {INPUT_SIZE}x{INPUT_SIZE}")
    print(f"\nResized dataset: {DATASET_DIR}")
else:
    print(f"Resized dataset already at: {DATASET_DIR}")

# Set up paths for training
train_img_dir = DATASET_DIR / 'images' / 'train'
val_img_dir   = DATASET_DIR / 'images' / 'val'
train_lbl_dir = DATASET_DIR / 'labels' / 'train'
val_lbl_dir   = DATASET_DIR / 'labels' / 'val'

train_images = sorted(train_img_dir.glob('*.jpg'))
val_images   = sorted(val_img_dir.glob('*.jpg'))

print(f"\nTraining dataset ({INPUT_SIZE}x{INPUT_SIZE}):")
print(f"  Train images: {len(train_images)}")
print(f"  Val images:   {len(val_images)}")

# Quick sanity check
sample_img = Image.open(train_images[0])
print(f"  Sample size: {sample_img.size}")

## 4. Build Darknet

In [ ]:
%%time

DARKNET_DIR = Path('/content/darknet')

if not (DARKNET_DIR / 'darknet').exists():
    # Install OpenCV dev libraries (needed for darknet build)
    !apt-get update -qq && apt-get install -y -qq libopencv-dev >/dev/null 2>&1

    # Clone AlexeyAB darknet
    if not (DARKNET_DIR / 'Makefile').exists():
        !git clone https://github.com/AlexeyAB/darknet.git {DARKNET_DIR}

    %cd {DARKNET_DIR}

    # Enable GPU + CUDNN + OpenCV (no CUDNN_HALF)
    !sed -i 's/OPENCV=0/OPENCV=1/' Makefile
    !sed -i 's/GPU=0/GPU=1/' Makefile
    !sed -i 's/CUDNN=0/CUDNN=1/' Makefile
    !sed -i 's/LIBSO=0/LIBSO=1/' Makefile

    # Fix CUDA toolkit > driver version mismatch on Colab:
    # Detect GPU compute capability and force SASS-only compilation (no PTX).
    import subprocess as _sp
    _gpu_info = _sp.check_output(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        text=True
    ).strip().split('\n')[0]
    _cc = _gpu_info.replace('.', '')  # e.g. "7.5" -> "75"
    print(f"Detected GPU compute capability: {_gpu_info} (sm_{_cc})")

    # The ARCH= block in the Makefile spans multiple continuation lines.
    # Comment out the entire block and append a clean single-line replacement.
    makefile = Path('Makefile')
    lines = makefile.read_text().splitlines()
    new_lines = []
    in_arch_block = False
    for line in lines:
        if line.startswith('ARCH='):
            in_arch_block = True
            new_lines.append('# PATCHED: ' + line)
        elif in_arch_block:
            new_lines.append('# PATCHED: ' + line)
            # End of continuation block when line doesn't end with backslash
            if not line.rstrip().endswith('\\'):
                in_arch_block = False
                # Insert our replacement right after the commented block
                new_lines.append(f'ARCH= -gencode arch=compute_{_cc},code=sm_{_cc}')
        else:
            new_lines.append(line)
    makefile.write_text('\n'.join(new_lines) + '\n')
    print(f"Patched Makefile ARCH to: -gencode arch=compute_{_cc},code=sm_{_cc}")

    !make clean 2>/dev/null; make -j$(nproc)
    %cd /content

    # Verify the binary was actually created
    if (DARKNET_DIR / 'darknet').exists():
        print("\nDarknet built successfully!")
    else:
        raise RuntimeError("Darknet build FAILED — binary not found. Check errors above.")
else:
    print("Darknet already built.")

# Verify
!{DARKNET_DIR}/darknet version 2>/dev/null || echo "Darknet binary ready"

## 5. Download Pre-trained Weights

In [ ]:
# Download YOLOv4-Tiny pre-trained weights (on COCO) for transfer learning
PRETRAINED_WEIGHTS = DARKNET_DIR / 'yolov4-tiny.conv.29'

if not PRETRAINED_WEIGHTS.exists():
    !wget -q https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v4_pre/yolov4-tiny.conv.29 \
        -O {PRETRAINED_WEIGHTS}
    print(f"Downloaded: {PRETRAINED_WEIGHTS}")
else:
    print(f"Weights already exist: {PRETRAINED_WEIGHTS}")

print(f"Size: {PRETRAINED_WEIGHTS.stat().st_size / 1024 / 1024:.1f} MB")

## 6. Generate Darknet Config Files

In [ ]:
# Config directory
CFG_DIR = PROJECT_DIR / 'cfg'
CFG_DIR.mkdir(parents=True, exist_ok=True)

# --- obj.names ---
names_path = CFG_DIR / 'obj.names'
with open(names_path, 'w') as f:
    for name in CLASS_NAMES:
        f.write(f"{name}\n")
print(f"Created: {names_path}")

# --- train.txt and val.txt (list of image paths) ---
train_txt = CFG_DIR / 'train.txt'
val_txt   = CFG_DIR / 'val.txt'

with open(train_txt, 'w') as f:
    for p in train_images:
        f.write(f"{p}\n")

with open(val_txt, 'w') as f:
    for p in val_images:
        f.write(f"{p}\n")

print(f"Created: {train_txt} ({len(train_images)} images)")
print(f"Created: {val_txt} ({len(val_images)} images)")

# --- obj.data ---
BACKUP_DIR = PROJECT_DIR / 'backup'
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

data_path = CFG_DIR / 'obj.data'
with open(data_path, 'w') as f:
    f.write(f"classes = {NUM_CLASSES}\n")
    f.write(f"train = {train_txt}\n")
    f.write(f"valid = {val_txt}\n")
    f.write(f"names = {names_path}\n")
    f.write(f"backup = {BACKUP_DIR}\n")
print(f"Created: {data_path}")

In [ ]:
# --- yolov4-tiny custom .cfg ---
# YOLOv4-Tiny config adapted for custom training
# Key formulas:
#   filters = (num_classes + 5) * 3
#   max_batches = max(num_classes * 2000, num_train_images, 6000)
#   steps = 80% and 90% of max_batches

filters = (NUM_CLASSES + 5) * 3
max_batches = max(NUM_CLASSES * 2000, len(train_images), 6000)
steps_80 = int(max_batches * 0.8)
steps_90 = int(max_batches * 0.9)

print(f"filters = {filters}")
print(f"max_batches = {max_batches}")
print(f"steps = {steps_80},{steps_90}")

cfg_content = f"""[net]
batch=64
subdivisions=16
width={INPUT_SIZE}
height={INPUT_SIZE}
channels=3
momentum=0.9
decay=0.0005
angle=0
saturation=1.5
exposure=1.5
hue=.1

learning_rate=0.00261
burn_in=1000
max_batches={max_batches}
policy=steps
steps={steps_80},{steps_90}
scales=.1,.1

mosaic=1

[convolutional]
batch_normalize=1
filters=32
size=3
stride=2
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=64
size=3
stride=2
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=64
size=3
stride=1
pad=1
activation=leaky

[route]
layers=-1
groups=2
group_id=1

[convolutional]
batch_normalize=1
filters=32
size=3
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=32
size=3
stride=1
pad=1
activation=leaky

[route]
layers=-1,-2

[convolutional]
batch_normalize=1
filters=64
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-6,-1

[maxpool]
size=2
stride=2

[convolutional]
batch_normalize=1
filters=128
size=3
stride=1
pad=1
activation=leaky

[route]
layers=-1
groups=2
group_id=1

[convolutional]
batch_normalize=1
filters=64
size=3
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=64
size=3
stride=1
pad=1
activation=leaky

[route]
layers=-1,-2

[convolutional]
batch_normalize=1
filters=128
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-6,-1

[maxpool]
size=2
stride=2

[convolutional]
batch_normalize=1
filters=256
size=3
stride=1
pad=1
activation=leaky

[route]
layers=-1
groups=2
group_id=1

[convolutional]
batch_normalize=1
filters=128
size=3
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=128
size=3
stride=1
pad=1
activation=leaky

[route]
layers=-1,-2

[convolutional]
batch_normalize=1
filters=256
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-6,-1

[maxpool]
size=2
stride=2

[convolutional]
batch_normalize=1
filters=512
size=3
stride=1
pad=1
activation=leaky

##################################

[convolutional]
batch_normalize=1
filters=256
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=512
size=3
stride=1
pad=1
activation=leaky

[convolutional]
size=1
stride=1
pad=1
filters={filters}
activation=linear

[yolo]
mask=3,4,5
anchors=10,14, 23,27, 37,58, 81,82, 135,169, 344,319
classes={NUM_CLASSES}
num=6
jitter=.3
scale_x_y=1.05
cls_normalizer=1.0
iou_normalizer=0.07
iou_loss=ciou
ignore_thresh=.7
truth_thresh=1
random=0
nms_kind=greedynms
beta_nms=0.6

[route]
layers=-4

[convolutional]
batch_normalize=1
filters=128
size=1
stride=1
pad=1
activation=leaky

[upsample]
stride=2

[route]
layers=-1,23

[convolutional]
batch_normalize=1
filters=256
size=3
stride=1
pad=1
activation=leaky

[convolutional]
size=1
stride=1
pad=1
filters={filters}
activation=linear

[yolo]
mask=0,1,2
anchors=10,14, 23,27, 37,58, 81,82, 135,169, 344,319
classes={NUM_CLASSES}
num=6
jitter=.3
scale_x_y=1.05
cls_normalizer=1.0
iou_normalizer=0.07
iou_loss=ciou
ignore_thresh=.7
truth_thresh=1
random=0
nms_kind=greedynms
beta_nms=0.6
"""

cfg_path = CFG_DIR / 'yolov4-tiny-deepoo.cfg'
with open(cfg_path, 'w') as f:
    f.write(cfg_content)

print(f"Created: {cfg_path}")

## 7. Verify Darknet Labels

Darknet expects `.txt` label files next to images (or in a `labels/` folder mirroring `images/`).  
Our dataset uses `images/train/` + `labels/train/` structure — darknet handles this automatically  
if we replace `images` with `labels` in the path.

In [ ]:
# Verify labels exist for training images
missing = 0
for img_path in train_images[:10]:  # sample check
    lbl_path = str(img_path).replace('/images/', '/labels/').replace('.jpg', '.txt')
    if not os.path.exists(lbl_path):
        missing += 1
        print(f"  Missing: {lbl_path}")

if missing == 0:
    print("All sampled labels found!")

# Show a sample label
sample_lbl = str(train_images[0]).replace('/images/', '/labels/').replace('.jpg', '.txt')
print(f"\nSample label ({sample_lbl}):")
with open(sample_lbl) as f:
    print(f.read()[:300])

In [ ]:
# Darknet needs labels co-located with images OR in labels/ folder.
# The standard YOLO txt format is: class_id cx cy w h (normalized)
# Our dataset already has labels in labels/ mirroring images/ — darknet
# automatically maps images/X.jpg -> labels/X.txt.
# But some darknet versions need labels beside images. Let's symlink to be safe.

for split in ['train', 'val']:
    img_dir = DATASET_DIR / 'images' / split
    lbl_dir = DATASET_DIR / 'labels' / split
    count = 0
    for lbl_file in lbl_dir.glob('*.txt'):
        target = img_dir / lbl_file.name
        if not target.exists():
            os.symlink(lbl_file, target)
            count += 1
    print(f"[{split}] Symlinked {count} label files into image directory")

## 8. Train YOLOv4-Tiny

In [ ]:
# Optional: If you want to resume training from a checkpoint on Drive, uncomment:
# RESUME_WEIGHTS = DRIVE_OUTPUT / 'yolov4-tiny-deepoo_last.weights'
# if RESUME_WEIGHTS.exists():
#     shutil.copy(RESUME_WEIGHTS, BACKUP_DIR / 'yolov4-tiny-deepoo_last.weights')
#     print(f"Copied checkpoint for resume: {RESUME_WEIGHTS}")

In [ ]:
import subprocess
import re
from tqdm.notebook import tqdm
from IPython.display import display, HTML

# Train with live tqdm progress bar
cmd = [
    str(DARKNET_DIR / 'darknet'), 'detector', 'train',
    str(data_path), str(cfg_path), str(PRETRAINED_WEIGHTS),
    '-dont_show', '-map', '-clear'
]

pbar = tqdm(total=max_batches, desc="Training", unit="iter",
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')

best_loss = float('inf')
last_map = None
current_iter = 0

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True, bufsize=1
)

try:
    for line in process.stdout:
        line = line.strip()

        # Parse iteration lines: "2000: 1.234, 1.567 avg loss, ..."
        iter_match = re.match(r'^\s*(\d+):\s+([\d.]+),\s+([\d.]+)\s+avg', line)
        if iter_match:
            iteration = int(iter_match.group(1))
            loss = float(iter_match.group(2))
            avg_loss = float(iter_match.group(3))

            advance = iteration - current_iter
            if advance > 0:
                pbar.update(advance)
                current_iter = iteration

            best_loss = min(best_loss, avg_loss)
            postfix = {'avg_loss': f'{avg_loss:.4f}', 'best': f'{best_loss:.4f}'}
            if last_map is not None:
                postfix['mAP'] = f'{last_map:.2f}%'
            pbar.set_postfix(postfix)

        # Parse mAP lines: "... mean average precision (mAP@0.50) = 0.8765 ..."
        map_match = re.search(r'mean average precision.*?=\s*([\d.]+)', line)
        if map_match:
            last_map = float(map_match.group(1)) * 100
            postfix = {'avg_loss': f'{avg_loss:.4f}', 'best': f'{best_loss:.4f}', 'mAP': f'{last_map:.2f}%'}
            pbar.set_postfix(postfix)

        # Print saving/important messages
        if any(kw in line.lower() for kw in ['saving', 'best_loss', 'error', 'cuda']):
            tqdm.write(line)

except KeyboardInterrupt:
    process.terminate()
    print("\nTraining interrupted by user.")
finally:
    process.wait()
    pbar.close()

print(f"\nTraining complete! Final iteration: {current_iter}, Best avg loss: {best_loss:.4f}")
if last_map is not None:
    print(f"Last mAP@0.50: {last_map:.2f}%")

In [ ]:
# Copy best and last weights to Drive for safekeeping
best_weights = BACKUP_DIR / 'yolov4-tiny-deepoo_best.weights'
last_weights = BACKUP_DIR / 'yolov4-tiny-deepoo_last.weights'

for w in [best_weights, last_weights]:
    if w.exists():
        shutil.copy(w, DRIVE_OUTPUT / w.name)
        print(f"Saved to Drive: {DRIVE_OUTPUT / w.name} ({w.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"WARNING: {w} not found")

# Also copy the training chart
chart = DARKNET_DIR / 'chart.png'
if chart.exists():
    shutil.copy(chart, DRIVE_OUTPUT / 'training_chart.png')
    from IPython.display import display, Image as IPImage
    display(IPImage(filename=str(chart)))
else:
    print("Training chart not found")

## 9. Evaluate Darknet Model (mAP)

In [ ]:
# Compute mAP on validation set using best weights
WEIGHTS_PATH = best_weights if best_weights.exists() else last_weights
print(f"Evaluating: {WEIGHTS_PATH}")

!{DARKNET_DIR}/darknet detector map \
    {data_path} \
    {cfg_path} \
    {WEIGHTS_PATH} \
    -dont_show

In [ ]:
# Visual test on a few validation images
import random as rng
rng.seed(42)
test_imgs = rng.sample(val_images, min(4, len(val_images)))

for img_path in test_imgs:
    !{DARKNET_DIR}/darknet detector test \
        {data_path} \
        {cfg_path} \
        {WEIGHTS_PATH} \
        {img_path} \
        -dont_show -ext_output 2>&1 | grep -E '(^$|poo:)'
    
    # darknet saves result to predictions.jpg
    pred_img = DARKNET_DIR / 'predictions.jpg'
    if pred_img.exists():
        plt.figure(figsize=(8, 8))
        plt.imshow(Image.open(pred_img))
        plt.title(img_path.name)
        plt.axis('off')
        plt.show()

## 10. Convert to TFLite INT8

**Path**: darknet weights → ONNX (via `Tianxiaomo/pytorch-YOLOv4`) → TFLite INT8 (via `onnx2tf`)

The `hunglc007/tensorflow-yolov4-tflite` repo is incompatible with TF2.16+/Keras 3,  
so we use the proven darknet → ONNX → TFLite pipeline instead.

In [ ]:
# Install conversion dependencies
!pip install -q onnx onnxruntime onnxsim onnxscript torch torchvision

import torch
import onnx
print(f"PyTorch: {torch.__version__}")
print(f"ONNX: {onnx.__version__}")

In [ ]:
# Clone Tianxiaomo/pytorch-YOLOv4 for darknet2onnx conversion
PYTORCH_YOLO_DIR = Path('/content/pytorch-YOLOv4')

if not PYTORCH_YOLO_DIR.exists():
    !git clone https://github.com/Tianxiaomo/pytorch-YOLOv4.git {PYTORCH_YOLO_DIR}
    print(f"Cloned to: {PYTORCH_YOLO_DIR}")
else:
    print(f"Already exists: {PYTORCH_YOLO_DIR}")

In [ ]:
%%time

# Step 1: Convert darknet weights -> ONNX
# We use Tianxiaomo's Darknet model class directly and force the legacy
# TorchScript-based ONNX exporter (dynamo=False) which properly serializes weights.

import sys
sys.path.insert(0, str(PYTORCH_YOLO_DIR))
from tool.darknet2pytorch import Darknet as DarknetPT

ONNX_RAW = PROJECT_DIR / 'yolov4_tiny_deepoo.onnx'

if not ONNX_RAW.exists() or ONNX_RAW.stat().st_size < 1_000_000:
    # Remove any corrupted previous attempt
    if ONNX_RAW.exists():
        ONNX_RAW.unlink()

    # Load darknet model
    model = DarknetPT(str(cfg_path))
    model.load_weights(str(WEIGHTS_PATH))
    model.eval()
    print(f"Loaded darknet model: {sum(p.numel() for p in model.parameters())} parameters")

    # Create dummy input
    dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)

    # Force legacy TorchScript-based exporter (dynamo=False) to properly embed weights
    torch.onnx.export(
        model,
        dummy_input,
        str(ONNX_RAW),
        opset_version=11,
        input_names=['input'],
        output_names=['boxes', 'confs'],
        dynamic_axes=None,
        dynamo=False,  # force legacy exporter
    )
    print(f"ONNX model exported: {ONNX_RAW}")
else:
    print(f"ONNX already exists: {ONNX_RAW}")

size_mb = ONNX_RAW.stat().st_size / 1024 / 1024
print(f"Size: {size_mb:.2f} MB")

if size_mb < 1.0:
    raise RuntimeError(f"ONNX model too small ({size_mb:.2f} MB) — likely corrupted. Delete and retry.")

In [ ]:
%%time

# Step 2: Simplify ONNX model (removes redundant ops, folds constants)
ONNX_SIMPLE = PROJECT_DIR / 'yolov4_tiny_deepoo_sim.onnx'

if not ONNX_SIMPLE.exists():
    !python -m onnxsim {ONNX_RAW} {ONNX_SIMPLE}
    print(f"Simplified ONNX: {ONNX_SIMPLE}")
else:
    print(f"Simplified ONNX already exists: {ONNX_SIMPLE}")

print(f"Size: {ONNX_SIMPLE.stat().st_size / 1024 / 1024:.2f} MB")

# Verify ONNX model
model = onnx.load(str(ONNX_SIMPLE))
print(f"\nInputs:")
for inp in model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"  {inp.name}: {shape}")
print(f"Outputs:")
for out in model.graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  {out.name}: {shape}")

In [ ]:
%%time

# Step 3: Convert ONNX -> TF SavedModel + TFLite INT8 via onnx2tf
!pip install -q onnx2tf tensorflow ai-edge-litert sng4onnx onnx_graphsurgeon

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")

# Prepare calibration data for INT8 quantization
# onnx2tf uses a calibration data directory with .npy files
# Format: one .npy per input tensor, named like: <input_name>_<index>.npy
CALIB_DIR = PROJECT_DIR / 'calibration_data'
CALIB_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)
calib_images = random.sample(train_images, min(200, len(train_images)))
print(f"Preparing {len(calib_images)} calibration images...")

# The ONNX model input is NCHW (pytorch convention): [1, 3, 416, 416]
# onnx2tf will handle the transpose to NHWC internally
# We save calibration data in NCHW format to match the ONNX input
calib_batch = []
for img_path in tqdm(calib_images, desc="Calibration data"):
    img = Image.open(img_path).convert('RGB')
    img = img.resize((INPUT_SIZE, INPUT_SIZE))
    img_array = np.array(img, dtype=np.float32) / 255.0
    # HWC -> CHW for ONNX input format
    img_array = np.transpose(img_array, (2, 0, 1))
    calib_batch.append(img_array)

calib_npy = CALIB_DIR / 'calib_data.npy'
np.save(calib_npy, np.array(calib_batch))
print(f"Calibration data: {calib_npy} (shape: {np.array(calib_batch).shape})")

In [ ]:
%%time

# Convert ONNX -> TF SavedModel -> TFLite INT8
# -dgc: disable accuracy correction (avoids crash on YOLO detection head Div ops)
# -osd: output SavedModel directory
SAVED_MODEL_DIR = PROJECT_DIR / 'saved_model'

# Clean previous failed attempt
if SAVED_MODEL_DIR.exists():
    shutil.rmtree(SAVED_MODEL_DIR)

!onnx2tf -i {ONNX_SIMPLE} \
    -o {SAVED_MODEL_DIR} \
    -osd \
    -dgc

# Verify SavedModel was created
sm_pb = SAVED_MODEL_DIR / 'saved_model.pb'
if not sm_pb.exists():
    raise RuntimeError("onnx2tf failed — no saved_model.pb generated. Check errors above.")
print(f"\nSavedModel created: {SAVED_MODEL_DIR}")
!ls -lh {SAVED_MODEL_DIR}

# Now convert SavedModel -> TFLite INT8 with full quantization
def representative_dataset_gen():
    """Generator yielding calibration data in NHWC format for TFLite quantization."""
    for img_path in calib_images:
        img = Image.open(img_path).convert('RGB')
        img = img.resize((INPUT_SIZE, INPUT_SIZE))
        img_array = np.array(img, dtype=np.float32) / 255.0
        yield [np.expand_dims(img_array, axis=0)]

converter = tf.lite.TFLiteConverter.from_saved_model(str(SAVED_MODEL_DIR))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.float32

print("Converting to TFLite INT8 (this may take a few minutes)...")
tflite_model = converter.convert()

TFLITE_PATH = PROJECT_DIR / 'deepoo_yolov4_tiny_416_int8.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_mb = TFLITE_PATH.stat().st_size / 1024 / 1024
print(f"\nTFLite INT8 model saved: {TFLITE_PATH}")
print(f"Size: {size_mb:.2f} MB")

## 11. Validate TFLite Model

In [ ]:
# Load and inspect TFLite model
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("=== TFLite Model Info ===")
print(f"\nInputs ({len(input_details)}):")
for d in input_details:
    print(f"  name={d['name']}, shape={d['shape']}, dtype={d['dtype']}")
    if 'quantization_parameters' in d:
        qp = d['quantization_parameters']
        if len(qp.get('scales', [])) > 0:
            print(f"    quantization: scale={qp['scales'][0]:.6f}, zero_point={qp['zero_points'][0]}")

print(f"\nOutputs ({len(output_details)}):")
for d in output_details:
    print(f"  name={d['name']}, shape={d['shape']}, dtype={d['dtype']}")

In [ ]:
# Run inference on a test image
def run_tflite_inference(interpreter, image_path, input_size=416):
    """Run TFLite inference and return raw outputs."""
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Preprocess
    img = Image.open(image_path).convert('RGB')
    img_resized = img.resize((input_size, input_size))
    img_array = np.array(img_resized, dtype=np.float32) / 255.0

    # Handle quantized input
    input_dtype = input_details[0]['dtype']
    if input_dtype == np.uint8:
        qp = input_details[0]['quantization_parameters']
        scale = qp['scales'][0]
        zero_point = qp['zero_points'][0]
        img_array = (img_array / scale + zero_point).astype(np.uint8)

    input_data = np.expand_dims(img_array, axis=0)

    # Run
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    # Get outputs
    outputs = []
    for od in output_details:
        out = interpreter.get_tensor(od['index'])
        outputs.append(out)

    return outputs, img


# Test on validation images
test_img = val_images[0]
outputs, orig_img = run_tflite_inference(interpreter, test_img)

print(f"Test image: {test_img.name}")
for i, out in enumerate(outputs):
    print(f"Output {i}: shape={out.shape}, dtype={out.dtype}, range=[{out.min():.4f}, {out.max():.4f}]")

In [ ]:
# YOLOv4-Tiny TFLite output post-processing
def decode_yolov4_tflite_output(outputs, input_size=416, conf_thresh=0.25, nms_thresh=0.45):
    """
    Decode YOLOv4-Tiny TFLite outputs into bounding boxes.

    After onnx2tf conversion, the TFLite model outputs:
      output[0]: [1, 2535, 1]    - confidence scores (per class)
      output[1]: [1, 2535, 1, 4] - bounding boxes (x1, y1, x2, y2)
    The extra dim=1 is because NUM_CLASSES=1.
    """
    # Identify which output is scores and which is boxes by shape
    if outputs[0].shape[-1] == 4 or (len(outputs[0].shape) == 4 and outputs[0].shape[-1] == 4):
        boxes_raw = outputs[0]
        scores_raw = outputs[1]
    else:
        scores_raw = outputs[0]
        boxes_raw = outputs[1]

    # Squeeze batch and extra class dims
    # boxes: [1, N, 1, 4] or [1, N, 4] -> [N, 4]
    boxes = np.squeeze(boxes_raw)
    if boxes.ndim == 1:
        boxes = boxes.reshape(1, -1)
    if boxes.ndim == 3:
        boxes = boxes.reshape(-1, 4)

    # scores: [1, N, num_classes] or [1, N, 1] -> [N, num_classes]
    scores = np.squeeze(scores_raw, axis=0)  # [N, num_classes]
    if scores.ndim == 1:
        scores = scores.reshape(-1, 1)

    # Get best class per box
    class_ids = np.argmax(scores, axis=-1)
    confidences = np.max(scores, axis=-1)

    # Filter by confidence
    mask = confidences > conf_thresh
    filtered_boxes = boxes[mask]
    filtered_scores = confidences[mask]
    filtered_classes = class_ids[mask]

    if len(filtered_boxes) == 0:
        return np.array([]), np.array([]), np.array([])

    # Ensure boxes are [N, 4] for NMS
    assert filtered_boxes.ndim == 2 and filtered_boxes.shape[1] == 4, \
        f"Unexpected box shape after filtering: {filtered_boxes.shape}"

    # NMS
    indices = tf.image.non_max_suppression(
        filtered_boxes, filtered_scores,
        max_output_size=50,
        iou_threshold=nms_thresh
    ).numpy()

    return filtered_boxes[indices], filtered_scores[indices], filtered_classes[indices]


# Visualize detections
def visualize_detections(image, boxes, scores, class_ids, class_names, input_size=416):
    """Draw detections on image."""
    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(image)

    w, h = image.size

    for box, score, cls_id in zip(boxes, scores, class_ids):
        x1, y1, x2, y2 = box
        # Boxes may be normalized (0-1) or in input_size pixel coords
        if x2 <= 1.5 and y2 <= 1.5:  # normalized
            x1, y1, x2, y2 = x1 * w, y1 * h, x2 * w, y2 * h
        else:  # input_size coords
            x1 = x1 * w / input_size
            y1 = y1 * h / input_size
            x2 = x2 * w / input_size
            y2 = y2 * h / input_size

        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                             linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        label = f"{class_names[int(cls_id)]}: {score:.2f}"
        ax.text(x1, y1 - 5, label, color='lime', fontsize=12,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))

    ax.set_title(f"Detections: {len(boxes)}")
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# Test on several validation images
import random as rng
rng.seed(123)
test_set = rng.sample(val_images, min(6, len(val_images)))

for img_path in test_set:
    outputs, orig_img = run_tflite_inference(interpreter, img_path)
    boxes, scores, class_ids = decode_yolov4_tflite_output(outputs, INPUT_SIZE)
    print(f"\n{img_path.name}: {len(boxes)} detections")
    visualize_detections(orig_img, boxes, scores, class_ids, CLASS_NAMES, INPUT_SIZE)

## 12. Benchmark TFLite Model

In [ ]:
import time

# Benchmark inference speed
n_runs = 50
input_details = interpreter.get_input_details()

# Create dummy input
input_dtype = input_details[0]['dtype']
if input_dtype == np.uint8:
    dummy = np.random.randint(0, 255, size=input_details[0]['shape'], dtype=np.uint8)
else:
    dummy = np.random.rand(*input_details[0]['shape']).astype(np.float32)

# Warmup
for _ in range(5):
    interpreter.set_tensor(input_details[0]['index'], dummy)
    interpreter.invoke()

# Benchmark
times = []
for _ in range(n_runs):
    t0 = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], dummy)
    interpreter.invoke()
    times.append(time.perf_counter() - t0)

times = np.array(times) * 1000  # ms
print(f"TFLite INT8 Inference Benchmark ({n_runs} runs):")
print(f"  Mean:   {times.mean():.2f} ms")
print(f"  Median: {np.median(times):.2f} ms")
print(f"  Std:    {times.std():.2f} ms")
print(f"  Min:    {times.min():.2f} ms")
print(f"  Max:    {times.max():.2f} ms")
print(f"  FPS:    {1000.0 / times.mean():.1f}")

## 13. Alternative: onnx2tf Direct TFLite Output

If the TFLite converter in Step 10 fails, `onnx2tf` can also produce TFLite  
files directly. Check the `saved_model/` directory — it may already contain  
`.tflite` files generated during the onnx2tf conversion.

In [ ]:
# Check if onnx2tf already produced TFLite files directly
import glob as _glob

onnx2tf_tflites = sorted(Path(SAVED_MODEL_DIR).glob('*.tflite')) if SAVED_MODEL_DIR.exists() else []
if onnx2tf_tflites:
    print("onnx2tf generated TFLite files directly:")
    for f in onnx2tf_tflites:
        print(f"  {f.name} ({f.stat().st_size / 1024 / 1024:.2f} MB)")
    # Use the integer-quantized one if available
    int8_candidates = [f for f in onnx2tf_tflites if 'int8' in f.name.lower() or 'integer' in f.name.lower()]
    if int8_candidates:
        alt_tflite = int8_candidates[0]
    else:
        alt_tflite = onnx2tf_tflites[-1]  # use last (usually full integer)
    TFLITE_PATH_ALT = PROJECT_DIR / 'deepoo_yolov4_tiny_416_int8_onnx2tf.tflite'
    shutil.copy(alt_tflite, TFLITE_PATH_ALT)
    print(f"\nCopied alternative TFLite: {TFLITE_PATH_ALT}")
else:
    print("No direct TFLite output from onnx2tf. Use the converter output from Step 10.")

## 14. Save Final Model to Google Drive

In [ ]:
# Save TFLite model
drive_tflite = DRIVE_OUTPUT / TFLITE_PATH.name
shutil.copy(TFLITE_PATH, drive_tflite)
print(f"TFLite model: {drive_tflite}")
print(f"Size: {drive_tflite.stat().st_size / 1024 / 1024:.2f} MB")

# Save model config for Android app
model_config = {
    'model_type': 'yolov4_tiny',
    'input_size': INPUT_SIZE,
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'quantization': 'INT8',
    'input_dtype': 'uint8',
    'output_dtype': 'float32',
    'input_mean': 0.0,
    'input_std': 255.0,
    'anchors': [
        [10, 14], [23, 27], [37, 58],
        [81, 82], [135, 169], [344, 319]
    ],
    'anchor_masks': {
        'large': [3, 4, 5],
        'small': [0, 1, 2]
    },
    'conf_threshold': 0.25,
    'nms_threshold': 0.45
}

config_path = DRIVE_OUTPUT / 'model_config.json'
with open(config_path, 'w') as f:
    json.dump(model_config, f, indent=2)
print(f"Config: {config_path}")

# Save names file for Android
names_out = DRIVE_OUTPUT / 'labels.txt'
with open(names_out, 'w') as f:
    for name in CLASS_NAMES:
        f.write(f"{name}\n")
print(f"Labels: {names_out}")

print(f"\nAll files saved to: {DRIVE_OUTPUT}")
!ls -lh {DRIVE_OUTPUT}

## 15. Summary

### Files produced:
- `deepoo_yolov4_tiny_416_int8.tflite` — INT8 quantized model for Android
- `model_config.json` — model metadata (classes, anchors, thresholds)
- `labels.txt` — class names
- `yolov4-tiny-deepoo_best.weights` — darknet weights (for reference)

### Conversion path:
darknet `.weights` → ONNX (Tianxiaomo/pytorch-YOLOv4) → onnxsim → onnx2tf → TFLite INT8

### Android Integration Notes:
- **Input**: `uint8` tensor, shape `[1, 416, 416, 3]`, values `[0, 255]`
- **Output**: `float32` — bounding boxes + class scores
- **Preprocessing**: scale pixel values by `1/255.0` is handled by quantization params
- **Anchors**: Use the anchors from `model_config.json` for decoding
- **NMS**: Apply non-max suppression on decoded boxes client-side

### Why YOLOv4-Tiny works on Android TFLite:
- Uses only standard ops (Conv2D, MaxPool, LeakyReLU) — all TFLite-compatible
- No exotic ops like Mish activation or complex reshape patterns
- Proven INT8 quantization support via the ONNX → onnx2tf pipeline
- Small model size (~6 MB quantized) fits mobile constraints